# BingePlay — Streaming Analytics Project

**Name:** Mayuri Nilange  
**Batch:** Oct_2026 DS_DA  
**Organisation:** The Unlox Academy  
**Date:** 16 November 2026

## Project Overview

BingePlay is a streaming analytics project focused on analysing user behaviour, subscriptions, revenue, ratings, engagement and churn patterns using SQL and Python.

## Tools & Technologies
- MySQL
- MySQL Workbench
- Python
- Jupyter Notebook
- SQLAlchemy
- PyMySQL
- Pandas


In [20]:
%pip install pandas sqlalchemy pymysql

Note: you may need to restart the kernel to use updated packages.


In [21]:
import pandas as pd
from sqlalchemy import create_engine, URL
from getpass import getpass

In [3]:
from getpass import getpass
from sqlalchemy import create_engine, URL

password = getpass("Enter your MySQL password: ")

url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password=password,
    host="127.0.0.1",
    port=3306,
    database="bingeplay"
)

engine = create_engine(url)
print("Connection created!")


Enter your MySQL password:  ········


Connection created!


In [4]:
result = pd.read_sql("SELECT DATABASE() AS database_name;", engine)
print(result)



  database_name
0     bingeplay


In [5]:
tables = pd.read_sql("SHOW TABLES;", engine)
print(tables)

  Tables_in_bingeplay
0             ratings
1               shows
2       subscriptions
3               users
4      watch_sessions


In [6]:

result = pd.read_sql("DESCRIBE ratings;", engine)
print(result)


        Field         Type Null  Key Default Extra
0   rating_id          int   NO  PRI    None      
1     user_id  varchar(10)   NO  MUL    None      
2     show_id  varchar(10)   NO  MUL    None      
3       stars      tinyint   NO  MUL    None      
4  rated_date         date   NO         None      


In [7]:

query = """
SELECT
    COUNT(*) AS active_subscriptions,
    SUM(monthly_price_inr) AS total_monthly_revenue
FROM subscriptions
WHERE status = 'active'
  AND (
      end_date IS NULL
      OR end_date > '2024-06-30'
  );
"""

result = pd.read_sql(query, engine)
print(result)


   active_subscriptions  total_monthly_revenue
0                  2340               784260.0



## Q1 — Active Revenue

**Answer:**
- Active subscriptions: 2,340
- Total monthly recurring revenue: ₹7,84,260

**Interpretation:**
As of 30 June 2024, BingePlay had 2,340 active subscriptions generating ₹7,84,260 in monthly recurring revenue.


In [8]:

query = """
SELECT
    MONTH(signup_date) AS month,
    COUNT(*) AS signup_count
FROM users
WHERE YEAR(signup_date) = 2024
  AND MONTH(signup_date) BETWEEN 1 AND 6
GROUP BY MONTH(signup_date)
ORDER BY month;
"""

result = pd.read_sql(query, engine)
print(result)


   month  signup_count
0      1           350
1      2           400
2      3           500
3      4           550
4      5           600
5      6           600



## Q2 — Signup Momentum

**Answer:**
Monthly signup counts are displayed in the query output.

**Highest signup month:** 5,6

**Interpretation:**
This analysis shows the monthly growth in BingePlay's user base and identifies the month with the highest number of new registrations.


In [9]:

query = """
SELECT
    device_type,
    COUNT(*) AS total_sessions,
    SUM(watch_minutes) AS total_watch_minutes,
    ROUND(AVG(watch_minutes), 2) AS avg_watch_minutes,
    ROUND(
        SUM(CASE WHEN completed = 1 THEN 1 ELSE 0 END)
        * 100.0 / COUNT(*), 2
    ) AS completion_rate
FROM watch_sessions
WHERE user_id IS NOT NULL
GROUP BY device_type
ORDER BY total_sessions DESC;
"""

result = pd.read_sql(query, engine)
print(result)


  device_type  total_sessions  total_watch_minutes  avg_watch_minutes  \
0      Mobile           50172            1504355.0              29.98   
1          TV           27981             840595.0              30.04   
2      Laptop           15105             453434.0              30.02   
3      Tablet            7091             210733.0              29.72   

   completion_rate  
0            60.24  
1            59.98  
2            60.51  
3            59.79  



## Q3 — Device Analytics

**Answer:**
Device-wise session counts, total watch minutes, average watch minutes and completion rates are shown above.

**Interpretation:**
This analysis helps identify the most popular devices used by BingePlay customers and understand their viewing behaviour.


In [10]:
query = """
SELECT
    stars,
    COUNT(*) AS rating_count,
    ROUND(
        COUNT(*) * 100.0 / (SELECT COUNT(*) FROM ratings), 2
    ) AS percentage
FROM ratings
GROUP BY stars
ORDER BY stars;
"""

result = pd.read_sql(query, engine)
print(result)

   stars  rating_count  percentage
0      1           234        4.68
1      2           352        7.04
2      3           847       16.94
3      4          1781       35.62
4      5          1786       35.72


In [11]:
high_ratings = pd.read_sql("""
SELECT
    ROUND(
        SUM(CASE WHEN stars IN (4, 5) THEN 1 ELSE 0 END)
        * 100.0 / COUNT(*), 2
    ) AS four_five_star_percentage
FROM ratings;
""", engine)

print(high_ratings)

   four_five_star_percentage
0                      71.34


## Q4 — Rating Distribution

**Answer:**
- Total ratings: 5,000
- 1-star ratings: 234 (4.68%)
- 2-star ratings: 352 (7.04%)
- 3-star ratings: 847 (16.94%)
- 4-star ratings: 1,781 (35.62%)
- 5-star ratings: 1,786 (35.72%)

**Combined 4-star and 5-star ratings:** 71.34%

**Interpretation:**
71.34% of users gave 4-star or 5-star ratings, indicating that the majority of users are satisfied with BingePlay content.

In [12]:

query = """
SELECT
    CASE
        WHEN is_original = 1 THEN 'BingePlay Originals'
        ELSE 'Acquired Content'
    END AS content_type,
    COUNT(*) AS total_shows,
    ROUND(AVG(imdb_rating), 2) AS average_imdb_rating
FROM shows
GROUP BY is_original;
"""

result = pd.read_sql(query, engine)
print(result)


          content_type  total_shows  average_imdb_rating
0     Acquired Content           70                 6.63
1  BingePlay Originals           30                 7.92



## Q5 — Originals vs Acquired

**Answer:**
The number of shows and average IMDb ratings for both content groups are displayed above
.

**Interpretation:**
This comparison helps determine whether BingePlay Originals or acquired content performs better based on average IMDb ratings.


In [13]:

query = """
WITH binge_days AS (
    SELECT
        user_id,
        show_id,
        session_date,
        COUNT(*) AS total_sessions
    FROM watch_sessions
    WHERE user_id IS NOT NULL
      AND session_date BETWEEN '2024-04-01' AND '2024-06-30'
    GROUP BY user_id, show_id, session_date
    HAVING COUNT(*) >= 5
)
SELECT
    COUNT(*) AS total_binge_days,
    (
        SELECT user_id
        FROM binge_days
        GROUP BY user_id
        ORDER BY COUNT(*) DESC, user_id
        LIMIT 1
    ) AS top_user_id,
    (
        SELECT COUNT(*)
        FROM binge_days
        WHERE user_id = (
            SELECT user_id
            FROM binge_days
            GROUP BY user_id
            ORDER BY COUNT(*) DESC, user_id
            LIMIT 1
        )
    ) AS top_user_binge_days
FROM binge_days;
"""

result = pd.read_sql(query, engine)
print(result)


   total_binge_days top_user_id  top_user_binge_days
0               414      U00420                    2



## Q6 — Binge Day Detection

**Answer:**
- Total binge days: 414
- Top user ID: U00420
- Binge days by top user: 2

**Interpretation:**
This analysis identifies users who repeatedly watch the same show on a single day. It helps BingePlay understand binge-watching behaviour and user engagement.


In [14]:
query = """
SELECT
    COUNT(*) AS total_q1_signups,
    SUM(
        CASE
            WHEN NOT EXISTS (
                SELECT 1
                FROM watch_sessions ws
                WHERE ws.user_id = u.user_id
            )
            THEN 1
            ELSE 0
        END
    ) AS never_watched_users
FROM users u
WHERE signup_date >= '2024-01-01'
  AND signup_date < '2024-04-01';
"""

result = pd.read_sql(query, engine)
print(result)


   total_q1_signups  never_watched_users
0              1250                226.0


## Q7 — Q1 Signups Who Never Watched

**Interpretation:**

This analysis identifies users who registered during Q1 2024 but never watched any show on BingePlay. It helps identify users who signed up but did not engage with the platform.


In [15]:
query = """
WITH current_plans AS (
    SELECT
        user_id,
        plan,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY start_date DESC
        ) AS rn
    FROM subscriptions
    WHERE status = 'active'
      AND (end_date IS NULL OR end_date > '2024-06-30')
)
SELECT
    COUNT(*) AS overpaying_users
FROM current_plans cp
WHERE cp.rn = 1
  AND cp.plan IN ('Premium', 'Family')
  AND EXISTS (
      SELECT 1
      FROM watch_sessions ws
      WHERE ws.user_id = cp.user_id
  )
  AND NOT EXISTS (
      SELECT 1
      FROM watch_sessions ws
      JOIN shows s
        ON ws.show_id = s.show_id
      WHERE ws.user_id = cp.user_id
        AND s.min_plan IN ('Premium', 'Family')
  );
"""

result = pd.read_sql(query, engine)
print(result)


   overpaying_users
0                 7



## Q8 — Over-paying Premium/Family Users

**Interpretation:**

This analysis identifies Premium and Family subscribers who have watched only Basic-tier content. These users may be suitable targets for downgrade offers.


In [16]:
query = """
WITH ranked_subscriptions AS (
    SELECT
        user_id,
        plan,
        start_date,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY start_date, subscription_id
        ) AS rn
    FROM subscriptions
),

first_plans AS (
    SELECT
        user_id,
        plan AS first_plan,
        start_date AS first_start_date
    FROM ranked_subscriptions
    WHERE rn = 1
),

first_upgrades AS (
    SELECT
        fp.user_id,
        MIN(s.start_date) AS first_upgrade_date
    FROM first_plans fp
    JOIN subscriptions s
        ON fp.user_id = s.user_id
    WHERE fp.first_plan = 'Basic'
      AND s.start_date > fp.first_start_date
      AND (
          CASE s.plan
              WHEN 'Basic' THEN 1
              WHEN 'Premium' THEN 2
              WHEN 'Family' THEN 3
          END
      ) > 1
    GROUP BY fp.user_id
),

active_users AS (
    SELECT DISTINCT user_id
    FROM subscriptions
    WHERE status = 'active'
      AND (end_date IS NULL OR end_date > '2024-06-30')
)

SELECT
    COUNT(*) AS total_upgrade_users,
    ROUND(
        AVG(DATEDIFF(fu.first_upgrade_date, u.signup_date)),
        2
    ) AS average_days_to_upgrade
FROM users u
JOIN first_plans fp
    ON u.user_id = fp.user_id
JOIN first_upgrades fu
    ON u.user_id = fu.user_id
JOIN active_users au
    ON u.user_id = au.user_id
WHERE u.signup_date >= '2024-01-01'
  AND u.signup_date < '2024-02-01';
"""

result = pd.read_sql(query, engine)
print(result)


   total_upgrade_users  average_days_to_upgrade
0                   55                    64.96



## Q9 — Upgrade Success Cohort

**Interpretation:**

This analysis identifies January 2024 users who initially subscribed to the Basic plan, later upgraded to Premium or Family, and remained active as of 30 June 2024. The average days indicate how long users took to upgrade after signing up.


In [17]:
query = """
WITH comeback_events AS (
    SELECT DISTINCT
        ws1.user_id,
        ws1.show_id,
        ws1.session_date AS incomplete_date
    FROM watch_sessions ws1
    WHERE ws1.user_id IS NOT NULL
      AND ws1.completed = 0
      AND EXISTS (
          SELECT 1
          FROM watch_sessions ws2
          WHERE ws2.user_id = ws1.user_id
            AND ws2.show_id = ws1.show_id
            AND ws2.session_date BETWEEN
                DATE_ADD(ws1.session_date, INTERVAL 1 DAY)
                AND DATE_ADD(ws1.session_date, INTERVAL 7 DAY)
      )
),

show_comebacks AS (
    SELECT
        show_id,
        COUNT(*) AS comeback_count
    FROM comeback_events
    GROUP BY show_id
),

ranked_shows AS (
    SELECT
        show_id,
        comeback_count,
        ROW_NUMBER() OVER (
            ORDER BY comeback_count DESC, show_id
        ) AS rn
    FROM show_comebacks
)

SELECT
    (SELECT COUNT(*) FROM comeback_events)
        AS total_comeback_events,
    rs.show_id AS top_show_id,
    s.title AS top_show_title,
    rs.comeback_count AS top_show_comeback_count
FROM ranked_shows rs
JOIN shows s
    ON rs.show_id = s.show_id
WHERE rs.rn = 1;
"""
result = pd.read_sql(query, engine)
print(result)


   total_comeback_events top_show_id    top_show_title  \
0                   4345        S088  Rayalaseema Raga   

   top_show_comeback_count  
0                       64  



## Q10 — Cliffhanger Comebacks

**Interpretation:**

This analysis identifies users who returned to watch the same show within seven days after an incomplete viewing session. It also identifies the show generating the highest number of comeback events.


In [18]:
query = """
WITH weekly_activity AS (
    SELECT DISTINCT
        user_id,
        DATE_SUB(
            session_date,
            INTERVAL WEEKDAY(session_date) DAY
        ) AS week_start
    FROM watch_sessions
    WHERE user_id IS NOT NULL
),

numbered_weeks AS (
    SELECT
        user_id,
        week_start,
        FLOOR(TO_DAYS(week_start) / 7) AS week_ordinal,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY week_start
        ) AS rn
    FROM weekly_activity
),

grouped_weeks AS (
    SELECT
        user_id,
        week_ordinal - rn AS streak_group
    FROM numbered_weeks
),

streaks AS (
    SELECT
        user_id,
        streak_group,
        COUNT(*) AS streak_length
    FROM grouped_weeks
    GROUP BY user_id, streak_group
)

SELECT
    COUNT(DISTINCT CASE
        WHEN streak_length >= 4 THEN user_id
    END) AS users_with_4_week_streak,

    MAX(streak_length) AS longest_streak_weeks,

    (
        SELECT user_id
        FROM streaks
        WHERE streak_length = (
            SELECT MAX(streak_length) FROM streaks
        )
        ORDER BY user_id
        LIMIT 1
    ) AS longest_streak_user_id

FROM streaks;
"""

result = pd.read_sql(query, engine)
print(result)


   users_with_4_week_streak  longest_streak_weeks longest_streak_user_id
0                      1675                    26                 U00213



## Q11 — Consecutive-week Engagement

**Answer:**
- Users with 4+ consecutive week streaks: 1675
- Longest streak: 26 weeks
- User with longest streak: U00213

**Interpretation:**

This analysis identifies consistently engaged users who watch BingePlay content across consecutive calendar weeks. It helps understand user retention and long-term engagement.


In [19]:
query = """
WITH monthly_watch AS (
    SELECT
        user_id,

        SUM(
            CASE
                WHEN session_date >= '2024-05-01'
                 AND session_date < '2024-06-01'
                THEN watch_minutes
                ELSE 0
            END
        ) AS may_minutes,

        SUM(
            CASE
                WHEN session_date >= '2024-06-01'
                 AND session_date < '2024-07-01'
                THEN watch_minutes
                ELSE 0
            END
        ) AS june_minutes

    FROM watch_sessions
    WHERE session_date >= '2024-05-01'
      AND session_date < '2024-07-01'
      AND user_id IS NOT NULL
    GROUP BY user_id
),

churn_calculation AS (
    SELECT
        user_id,
        may_minutes,
        june_minutes,

        ROUND(
            (may_minutes - june_minutes)
            * 100.0 / may_minutes, 2
        ) AS drop_percentage

    FROM monthly_watch
    WHERE may_minutes > 0
),

churn_users AS (
    SELECT
        cc.user_id,
        u.name,
        cc.may_minutes,
        cc.june_minutes,
        cc.drop_percentage

    FROM churn_calculation cc
    JOIN users u
        ON cc.user_id = u.user_id

    WHERE cc.drop_percentage >= 50
)

SELECT
    user_id,
    name,
    may_minutes,
    june_minutes,
    drop_percentage,
    COUNT(*) OVER () AS total_churn_users

FROM churn_users
ORDER BY drop_percentage DESC;
"""

result = pd.read_sql(query, engine)
print(result)


    user_id              name  may_minutes  june_minutes  drop_percentage  \
0    U00023    Amit Mukherjee         43.0           0.0           100.00   
1    U00166  Shaurya Malhotra         94.0           0.0           100.00   
2    U00211        Ravi Menon        336.0           0.0           100.00   
3    U00225     Kritika Patil        209.0           0.0           100.00   
4    U00237    Shaurya Bansal        126.0           0.0           100.00   
..      ...               ...          ...           ...              ...   
516  U01858  Sanjay Mukherjee        296.0         147.0            50.34   
517  U02530       Nandini Roy        451.0         224.0            50.33   
518  U01192   Rohit Mukherjee        136.0          68.0            50.00   
519  U01806      Yuvraj Raman         78.0          39.0            50.00   
520  U02100      Vikram Singh        204.0         102.0            50.00   

     total_churn_users  
0                  521  
1                  521  


## ## Q12 — Churn Signal Detection

**Answer:**
- Total churn signal users: 521

**Interpretation:**

This analysis identifies users whose watch time decreased by 50% or more in June compared with May 2024. These users may be at risk of leaving the platform and can be targeted with personalised retention offers.
